In [6]:
import numpy as np
import os
from PIL import Image
from predict_loop_functions import generate_noise, visual_prediction

#### stochastic_binarization code 

In [32]:
num_frames = 10
sigma = 3
scans = [[4,7]]

image1 = np.full(shape = (10,144,256), fill_value = 128)

def process_noise_seed(noise_type: str, image) -> np.array:
        """
        Parameters
        ----------
        noise_seeds: int
            noise seeds, same as entered into inner_predict_loop() to iterate for each noise seed in threadpoolexecutor
        
        Returns
        -------
        array
            concatenated object of all predictions for every scan and noise seed
        """
        if noise_type.lower() == 'stochastic binarization': 
            # code here
            image_prob = image / 256
            prob_results = np.random.binomial(1, image_prob)
            image = (prob_results * 255).astype('uint8')
            prediction1_array = [visual_prediction(pair[0], pair[1], image) for pair in scans]
            return np.concatenate(prediction1_array, axis = 1)
        else: noise_type_process = noise_type
        
        new_noise = generate_noise(noise_type_process, num_frames, sigma)
        new_image = (image + new_noise).astype('uint8')
        
        prediction1_array = [visual_prediction(pair[0], pair[1], new_image) for pair in scans]
        return np.concatenate(prediction1_array, axis = 1)

In [33]:
test = process_noise_seed('stochastic Binarization', image1)

In [34]:
test.shape

(10, 7493)

In [13]:
test

array([[3.2309022 , 0.6355303 , 0.9747958 , ..., 0.12141243, 0.08360946,
        0.08332483],
       [3.609784  , 0.76145893, 0.9482532 , ..., 0.10110712, 0.06514998,
        0.0746242 ],
       [3.4852033 , 0.92149115, 0.9993638 , ..., 0.0778225 , 0.04256436,
        0.05210532],
       ...,
       [1.5475359 , 0.7595892 , 0.6971378 , ..., 0.16558708, 0.07755589,
        0.07595568],
       [1.4612225 , 0.7331971 , 0.66443354, ..., 0.17317283, 0.08259723,
        0.0766041 ],
       [1.4291986 , 0.71346694, 0.6494955 , ..., 0.17448345, 0.08408111,
        0.07414293]], dtype=float32)

#### image examples

In [7]:
# for now assign to x random images
def random_images(num, train = True, path = os.path.join("//imagenet-mini")):
    """
    NOTE: directory structure for subfolers as follows
    --imagenet-mini
        -->train
            -->folder1
                -->picture1-1
                -->picture2-1.....
            -->folder2...
            -->folder3...
                -->picture1-3
                -->pictuer2-3
                -->picture....
        -->validation


    Parameters
    ----------
    num: int
        number of images 
    frames: int
        number of frames
    train: bool
        using training or validation set, defaults to train
    path: string
        defaults to cwd for training folder, otherwise path to train/val. folders needed

    Returns
    -------
    folder_id
        folder # used
    image_ids
        image #'s used
    np.stack(final_array)
        stack of num DIFFERENT images in one object, shape (num, 144, 256)
    """

    if train == True: train_val = '//train' 
    else: train_val = '//val'

    folder_path = path + train_val # get path to train/val folders
    initial_folders = os.listdir(folder_path)

    folder_id = np.random.randint(0, len(initial_folders) - 1) # get path to random folder within train/val
    image_folder_path = folder_path + '//' + initial_folders[folder_id]

    image_folder = os.listdir(image_folder_path) # path to folder in train/val with images

    image_ids = np.random.randint(0,len(image_folder) - 1, size = num)
    final_array = np.empty(shape = (num, 144, 256))

    for i in range(num):
        image_name = '//' + image_folder[i]
        image_path = image_folder_path + image_name
        image = Image.open(image_path)
        image = image.convert('L')
        if image.size != (256, 144): image = image.resize((256, 144))
        image = np.array(image)
        final_array[i] = image

    return folder_id, image_ids, np.stack(final_array, axis = 0)


In [8]:
folder_id, image_ids, image_stack = random_images(5, path = "C://Users//joshf//Downloads//imagenet-mini")

In [10]:
image_stack.shape

(5, 144, 256)

In [11]:
image_stack_zero = image_stack[0]

In [ ]:
image_zero = Image.fromarray(image_stack_zero)
image_zero.show()

<PIL.Image.Image image mode=F size=256x144>